In [ ]:
!pip install -q scikit-learn==1.5.2

In [ ]:
import ast
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

In [ ]:
COMP_PATH = Path("/kaggle/input/llm-classification-finetuning")

train_path = COMP_PATH / "train.csv"
test_path = COMP_PATH / "test.csv"
sample_path = COMP_PATH / "sample_submission.csv"

print(train_path.exists(), test_path.exists(), sample_path.exists())

In [ ]:
LABELS = ["winner_model_a", "winner_model_b", "winner_tie"]

def safe_parse_list_str(x):
    if not isinstance(x, str):
        return "" if pd.isna(x) else str(x)

    s = x.strip()
    if not s:
        return ""

    # Try JSON first
    try:
        val = json.loads(s)
        if isinstance(val, list) and len(val) > 0:
            return "" if val[0] is None else str(val[0])
        return str(val)
    except Exception:
        pass

    # Try python literal
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list) and len(val) > 0:
            return "" if val[0] is None else str(val[0])
        return str(val)
    except Exception:
        return s

def truncate_text(s, max_chars):
    s = "" if s is None else str(s)
    return s[:max_chars]

def make_input_text(prompt, ra, rb, max_prompt_chars=400, max_resp_chars=700):
    prompt = truncate_text(prompt, max_prompt_chars)
    ra = truncate_text(ra, max_resp_chars)
    rb = truncate_text(rb, max_resp_chars)
    return f"Prompt:\n{prompt}\n\nResponse A:\n{ra}\n\nResponse B:\n{rb}"

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

for c in ["prompt", "response_a", "response_b"]:
    train_df[c] = train_df[c].map(safe_parse_list_str)
    test_df[c] = test_df[c].map(safe_parse_list_str)

train_df["label_name"] = train_df[LABELS].idxmax(axis=1)
train_df["label"] = train_df["label_name"].map({
    "winner_model_a": 0,
    "winner_model_b": 1,
    "winner_tie": 2
}).astype(int)

print(train_df.shape, test_df.shape)
train_df.head(2)

In [ ]:
X_text = train_df.apply(
    lambda r: make_input_text(r["prompt"], r["response_a"], r["response_b"]),
    axis=1
).tolist()

X_test_text = test_df.apply(
    lambda r: make_input_text(r["prompt"], r["response_a"], r["response_b"]),
    axis=1
).tolist()

y = train_df["label"].values

print("Train texts:", len(X_text))
print("Test texts:", len(X_test_text))

In [ ]:
vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=50000,
    sublinear_tf=True,
)

X = vec.fit_transform(X_text)
X_test = vec.transform(X_test_text)

print(X.shape, X_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof = np.zeros((len(train_df), 3))
test_pred = np.zeros((len(test_df), 3))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Fold {fold}")

    clf = LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        solver="saga",
        C=3.0,
        n_jobs=-1,
        verbose=0,
    )

    clf.fit(X[tr_idx], y[tr_idx])

    val_pred = clf.predict_proba(X[va_idx])
    oof[va_idx] = val_pred
    test_pred += clf.predict_proba(X_test) / 3

    fold_ll = log_loss(y[va_idx], val_pred, labels=[0, 1, 2])
    print(f"  fold logloss: {fold_ll:.6f}")

overall_ll = log_loss(y, oof, labels=[0, 1, 2])
print(f"\nOOF logloss: {overall_ll:.6f}")

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"].astype(str),
    "winner_model_a": test_pred[:, 0],
    "winner_model_b": test_pred[:, 1],
    "winner_tie": test_pred[:, 2],
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.head()

In [ ]:
print(Path("/kaggle/working/submission.csv").exists())
print(pd.read_csv("/kaggle/working/submission.csv").shape)

In [ ]:
print("Submission file ready at /kaggle/working/submission.csv")